In [ ]:
using Pkg
Pkg.activate("./insight_calc")
using D4M, DataFrames, DotEnv
include("insight_calc/src/automata.jl")
include("insight_calc/src/services.jl")
include("insight_calc/src/storage.jl")

In [ ]:
Pkg.status()

In [ ]:
session = getServer()
fieldnames(typeof(session))

In [ ]:
listTables()

In [ ]:
srv     = getServer()
jobId   = _submitJob(srv.session, srv.project, """
    SELECT id, COUNT(*) AS n
    FROM `creator-d4m-2026-1774038056.insight_metadata.yt_metadata`
    GROUP BY id
    HAVING COUNT(*) > 1
    ORDER BY n DESC
""")
url     = "$_BQ_BASE/projects/$(srv.project)/queries/$jobId?maxResults=100&location=us-central1"
payload = JSON3.read(HTTP.get(url, _authHeaders(srv.session)).body)

isempty(get(payload, :rows, [])) ? "No duplicates" : begin
    cols = [String(f.name) for f in payload.schema.fields]
    rows = [[isnothing(c.v) ? "" : String(c.v) for c in row.f] for row in payload.rows]
    DataFrame(permutedims(hcat(rows...)), cols)
end


In [ ]:
srv     = getServer()
jobId   = _submitJob(srv.session, srv.project, """
    SELECT id, query_term
    FROM `creator-d4m-2026-1774038056.insight_metadata.yt_metadata`
""")
url     = "$_BQ_BASE/projects/$(srv.project)/queries/$jobId?maxResults=10000&location=us-central1"
payload = JSON3.read(HTTP.get(url, _authHeaders(srv.session)).body)


In [ ]:
using DataFrames

srv     = getServer()
jobId   = _submitJob(srv.session, srv.project, """
    SELECT id, qt AS query_term
    FROM `creator-d4m-2026-1774038056.insight_metadata.yt_metadata`,
    UNNEST(query_term) AS qt
""")
url     = "$_BQ_BASE/projects/$(srv.project)/queries/$jobId?maxResults=10000&location=us-central1"
payload = JSON3.read(HTTP.get(url, _authHeaders(srv.session)).body)

ids   = [String(row.f[1].v) for row in payload.rows]
terms = [String(row.f[2].v) for row in payload.rows]

aa = Assoc(ids, terms, fill("1", length(ids)))

all_ids   = unique(ids)
all_terms = unique(terms)
lookup    = Set(zip(ids, terms))

DataFrame(
    "id"       => all_ids,
    [t => [in((id, t), lookup) ? "1" : "" for id in all_ids] for t in all_terms]...
)


In [ ]:
using DataFrames

aa        = queryYtMetadata(":")
all_ids   = unique(getrow(aa))
all_terms = unique(getcol(aa))
lookup    = Set(zip(getrow(aa), getcol(aa)))

DataFrame(
    "id" => all_ids,
    [t => [in((id, t), lookup) ? "1" : "" for id in all_ids] for t in all_terms]...
)
